In [1]:
import pickle
import os
import json
import math
import torch
import random
import hashlib
import argparse

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
from copy import deepcopy
import scipy.sparse as sp
import torch.optim as optim
import torch.nn.functional as F
import torch.optim.lr_scheduler as lr_scheduler

from scipy.sparse import coo_matrix
from collections import defaultdict
from torch_scatter import scatter_sum
from torch_geometric.utils import softmax
from torch_geometric.data import Data, Batch
from lifelines.utils import concordance_index
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch_geometric.utils import dense_to_sparse
from torch_geometric.nn import GCNConv, GATConv,GraphNorm,global_mean_pool
from sklearn.metrics import roc_auc_score,average_precision_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
)


import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print('torch version: ', torch.__version__)
torch.cuda.set_device(1)
print(device)

torch version:  2.7.0.dev20250312+cu128
cuda:1


In [ ]:
def load_adj_matrices(folder_path, drop_suffix=True):
    adj_dict = {}
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):  # 只处理csv文件
            file_path = os.path.join(folder_path, filename)
            adj_matrix = pd.read_csv(file_path, index_col=0)  # 如果第一列是索引
            #adj_matrix = pd.read_csv(file_path, header=None)  # 如果没有表头
            key = os.path.splitext(filename)[0] if drop_suffix else filename
            
            #adj_dict[key] = adj_matrix.values  # 存为 numpy 数组
            adj_dict[key] = adj_matrix       # 如果想保留 DataFrame
            
    return adj_dict

def build_pathway_dict(folder_path, cnv_amp_file, cnv_del_file, snv_file):
    
    cnv_amp = pd.read_csv(cnv_amp_file, index_col=0)
    cnv_del = pd.read_csv(cnv_del_file, index_col=0)
    snv = pd.read_csv(snv_file, index_col=0)

    pathway_dict = {}

    for filename in os.listdir(folder_path):
        if not filename.endswith(".csv"):
            continue
        #print(filename)
        file_path = os.path.join(folder_path, filename)
        adj_matrix = pd.read_csv(file_path, index_col=0)
        genes_in_pathway = adj_matrix.index.tolist()

        sample_dict = {}
        for sample in cnv_amp.columns:  # 
            cnv_amp_vals = cnv_amp[sample].reindex(genes_in_pathway, fill_value=0)
            cnv_del_vals = cnv_del[sample].reindex(genes_in_pathway, fill_value=0) if sample in cnv_del.columns else pd.Series(0, index=genes_in_pathway)
            snv_vals = snv[sample].reindex(genes_in_pathway, fill_value=0) if sample in snv.columns else pd.Series(0, index=genes_in_pathway)
            df = pd.DataFrame({
                "CNV_amp": cnv_amp_vals,
                "CNV_del": cnv_del_vals,
                "SNV": snv_vals,
            })
            sample_dict[sample] = df
        key = os.path.splitext(filename)[0]
        pathway_dict[key] = sample_dict

    return pathway_dict

def build_pathway_dict_singleomic(folder_path, exp_file):

    exp_data = pd.read_csv(exp_file, index_col=0)

    pathway_dict = {}

    for filename in os.listdir(folder_path):
        if not filename.endswith(".csv"):
            continue
        #print(filename)
        file_path = os.path.join(folder_path, filename)
        adj_matrix = pd.read_csv(file_path, index_col=0)
        genes_in_pathway = adj_matrix.index.tolist()

        sample_dict = {}
        for sample in exp_data.columns:  # 
            exp_vals = exp_data[sample].reindex(genes_in_pathway, fill_value=0)
            df = pd.DataFrame({
                "Exp": exp_vals
            })
            sample_dict[sample] = df
        key = os.path.splitext(filename)[0]
        pathway_dict[key] = sample_dict

    return pathway_dict

def dot_product_decode(Z):
    return torch.sigmoid(torch.mm(Z, Z.t()))

def seed_everything(seed = 3078):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def save_exp_result(setting, result, dir_path):
    ''' Save result dictionaries as JSON file'''
    exp_name = setting['exp_name']
    #del setting['max_epoch']
    #del setting['train_batch_size']
    #del setting['test_batch_size']

    hash_key = hashlib.sha1(str(setting).encode()).hexdigest()[:6]
    filename = dir_path+'/{}-{}.json'.format(exp_name, hash_key)
    result.update(setting)
    with open(filename, 'w') as f:
        json.dump(result, f)

def sparse_to_tuple(sparse_mx):
    if not isinstance(sparse_mx, coo_matrix):
        sparse_mx = sparse_mx.tocoo()
    coords = np.vstack((sparse_mx.row, sparse_mx.col)).T  # (num_nonzero, 2)
    values = sparse_mx.data
    shape = sparse_mx.shape
    return coords, values, shape

class MultiHeadAttentionModule(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super(MultiHeadAttentionModule, self).__init__()
        # 定义多头注意力层
        self.multihead_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout,
                                                    batch_first=True)
        # 定义层归一化和全连接层
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        if isinstance(query, torch.sparse.Tensor):
            query = query.to_dense()

        if isinstance(key, torch.sparse.Tensor):
            key = key.to_dense()

        if isinstance(value, torch.sparse.Tensor):
            value = value.to_dense()

        # print(isinstance(value, torch.sparse.Tensor))

        # query, key, value 的维度应该是 [seq_len, batch_size, embed_dim]
        attn_output, attn_weights = self.multihead_attn(query, key, value, attn_mask=mask)
        # 跳跃连接 + 层归一化
        output = self.layer_norm(query + self.dropout(attn_output))
        # 输出经过全连接层
        output = self.fc(output)
        return output, attn_weights

def logging(msg, outdir, log_fpath):
    fpath = os.path.join(outdir, log_fpath)
    if not os.path.isdir(outdir):
        os.mkdir(outdir)
    with open(fpath, 'a') as fw:
        fw.write("%s\n" % msg)
    print(msg)

def log_learning_rates(optimizer, outdir, filename='lr.log'):
    lr_info = " | ".join(
        f"Group {i}: {group['lr']:.4e}" 
        for i, group in enumerate(optimizer.param_groups[:-1])  # 不包含loss_fn组
    )
    logging(f'LR after update: {lr_info}', outdir, filename)

def build_sample_features(big_dict, pathway_order):
    
    sample_features = {}
    samples = list(next(iter(big_dict.values())).keys())

    for sample in samples:
        features = []
        for pathway in pathway_order:
            features.append(big_dict[pathway][sample])
        sample_features[sample] = torch.stack(features)
    return sample_features

class TwoLForwardNetwork(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, dropout_pathway = 0.1,
                 num_class = 5,activation_func=nn.ReLU(),out_activation=None):
        super(TwoLForwardNetwork, self).__init__()
        
        self.layers = nn.Sequential(
             nn.Linear(input_dim, hidden_dim1).cuda(),
             nn.BatchNorm1d(hidden_dim1),
             #nn.LayerNorm(hidden_dim1),
             activation_func,
             nn.Dropout(dropout_pathway),
             nn.Linear(hidden_dim1, hidden_dim2).cuda(),
             nn.BatchNorm1d(hidden_dim2),
             #nn.LayerNorm(hidden_dim2),
             activation_func,
             nn.Dropout(dropout_pathway),
             nn.Linear(hidden_dim2, num_class)).cuda()
        
        self.out_activation = out_activation

    def forward(self, x):
        if self.out_activation:
            return self.out_activation(self.layers(x))
        else:
            return self.layers(x)

def split_dict(sample_dict, ratios=(0.8, 0.1, 0.1), seed=666):
    keys = list(sample_dict.keys())
    random.seed(seed)
    random.shuffle(keys)  # 打乱顺序
    
    n = len(keys)
    n_train = int(ratios[0] * n)
    n_val = int(ratios[1] * n)
    
    train_keys = keys[:n_train]
    val_keys = keys[n_train:n_train + n_val]
    test_keys = keys[n_train + n_val:]
    
    train_dict = {k: sample_dict[k] for k in train_keys}
    val_dict = {k: sample_dict[k] for k in val_keys}
    test_dict = {k: sample_dict[k] for k in test_keys}
    
    return train_dict, val_dict, test_dict

def adj_df_to_edge_index(adj_df: pd.DataFrame):
    """把DataFrame邻接矩阵转成edge_index"""
    adj = adj_df.values
    row, col = np.nonzero(adj)
    edge_index = torch.tensor([row, col], dtype=torch.long)
    return edge_index, adj_df.index.tolist()  # 同时返回基因名顺序

class ExpressionDataset(Dataset):
    def __init__(self, expr_df: pd.DataFrame, labels: dict):
        self.expr_df = expr_df
        self.labels = labels
        self.sample_names = list(labels.keys())

    def __len__(self):
        return len(self.sample_names)

    def __getitem__(self, idx):
        sample_name = self.sample_names[idx]
        label = self.labels[sample_name]
        return {
            'sample_name': sample_name,
            'label': torch.tensor(label, dtype=torch.long)
        }

def collate_fn(batch):
    sample_names = [item['sample_name'] for item in batch]
    labels = torch.stack([item['label'] for item in batch])
    return {
        'sample_names': sample_names,
        'labels': labels
    }

class GenePathwayProcessor:
    def __init__(self, pathways: dict):
        """预处理通路信息（仅需初始化一次）"""
        self.pathway_info = self._preprocess_pathways(pathways)
        # 提取所有通路的基因，构建全局基因索引（加速表达量查询）
        all_genes = list(set(g for p in self.pathway_info for g in p["genes"]))
        self.gene2idx = {g: i for i, g in enumerate(all_genes)}
        self.num_genes = len(all_genes)

    def _preprocess_pathways(self, pathways: dict):
        """将通路的邻接矩阵转换为边索引，并记录通路包含的基因"""
        pathway_list = []
        for p_name, adj_df in pathways.items():
            # 邻接矩阵→边索引（稀疏表示，节省内存）
            adj_matrix = adj_df.values  # 转为numpy矩阵
            edge_index, _ = dense_to_sparse(torch.tensor(adj_matrix, dtype=torch.float32))
            # 记录通路包含的基因（邻接矩阵的行/列名）
            pathway_genes = adj_df.index.tolist()
            pathway_list.append({
                "name": p_name,
                "edge_index": edge_index,  # [2, E]，E为边数
                "genes": pathway_genes,    # 通路包含的基因列表
                "num_genes": len(pathway_genes)
            })
        return pathway_list

    def process_batch(self, expr_df: pd.DataFrame, sample_names: list):

        num_samples = len(sample_names)
        
        # ----------------------
        # 1. 批量提取基因表达量（核心优化）
        # ----------------------
        # 构建样本×基因的表达矩阵（缺失值填0）
        expr_matrix = expr_df.reindex(self.gene2idx.keys())[sample_names].fillna(0.0).T
        expr_tensor = torch.tensor(expr_matrix.values, dtype=torch.float32)  # [N_samples, N_genes]
        
        # ----------------------
        # 2. 构建每个样本的通路网络数据
        # ----------------------
        batch_list = []
        for sample_idx in range(num_samples):
            sample_data_list = []
            # 提取当前样本的基因表达量（[N_genes,]）
            sample_expr = expr_tensor[sample_idx]
            
            for p in self.pathway_info:
                # 通路基因在全局基因索引中的位置
                gene_indices = torch.tensor([self.gene2idx[g] for g in p["genes"]], dtype=torch.long)
                # 提取当前通路的基因表达量（[num_genes_in_pathway, 1]）
                x = sample_expr[gene_indices].unsqueeze(1)  # 保持特征维度为1
                
                # 构建通路网络数据（表达量+边索引）
                data = Data(
                    x=x,
                    edge_index=p["edge_index"],  # 复用预计算的边索引
                    num_nodes=p["num_genes"]
                )
                sample_data_list.append(data)
            
            # 将当前样本的所有通路打包成一个Batch
            sample_batch = Batch.from_data_list(sample_data_list)
            batch_list.append(sample_batch)
        
        # ----------------------
        # 3. 合并所有样本的Batch（实现多样本并行）
        # ----------------------
        total_batch = Batch.from_data_list(batch_list)
        return total_batch

class InterpretableTransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_ff=128, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            dropout=dropout,
            embed_dim=d_model,
            num_heads=nhead,
            batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.ReLU(),
            nn.Linear(dim_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.self_attn(
            x_norm, x_norm, x_norm,
            need_weights=True,
            average_attn_weights=False
        )
        x = x + self.dropout(attn_out)
        x_norm = self.norm2(x)
        ffn_out = self.ffn(x_norm)
        x = x + self.dropout(ffn_out)
        return x, attn_weights
    
class PathwayAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )

    def forward(self, x, batch):
        """
        Args:
            x: [Total_Genes_in_Batch, dim]
            batch: [Total_Genes_in_Batch] 记录每个基因属于哪个通路
        """
        # 1. 计算每个基因的原始得分
        raw_attn = self.gate(x)          # [Total_Genes_in_Batch, 1]
        
        # 2. 【关键】分段 Softmax
        # 它会确保同一个通路内的基因权重之和为 1，不同通路互不影响
        attn = softmax(raw_attn, batch)  # [Total_Genes_in_Batch, 1]
        
        # 3. 加权特征
        weighted_x = x * attn            # [Total_Genes_in_Batch, dim]
        
        # 4. 聚合得到通路特征
        # 相当于 global_sum_pool，但因为权重和为 1，本质上是加权平均
        pathway_feats = scatter_sum(weighted_x, batch, dim=0) 
        
        return pathway_feats,attn
    
class PathwayGATModel(torch.nn.Module):
    def __init__(self,
                 gene_in_dim: int = 1,          
                 gene_hidden_dim1: int = 8,    
                 gene_hidden_dim2: int = 32,    
                 pathway_in_dim: int = 32,      
                 pathway_hidden_dim1: int = 8,  
                 MLP_input_dim: int = 2504,   
                 MLP_hidden_dim1: int = 512,
                 MLP_hidden_dim2: int = 64 , 
                 num_class: int = 5,            
                 dropout_pathway: float = 0.5,
                 GAT_dropout: float = 0.5,
                 transformer_heads: int = 2,
                 transformer_layers: int = 3,
                 final_pathway_dim: int =32):
        super(PathwayGATModel, self).__init__()

        # ----------------------
        # 通路内的基因图卷积(GCN)
        # ----------------------
        self.GCN1 = GCNConv(gene_in_dim, gene_hidden_dim1,add_self_loops=True).cuda()
        self.GCN2 = GCNConv(gene_hidden_dim1, gene_hidden_dim2,add_self_loops=True).cuda()
        self.gene_pool1 = PathwayAttention(gene_hidden_dim1)
        self.gene_pool2 = PathwayAttention(gene_hidden_dim2)
        # ----------------------
        # 通路GAT（处理通路间网络）
        # ----------------------
        self.pathway_GAT1 = GATConv(in_channels=pathway_in_dim,
                                    out_channels=pathway_hidden_dim1,
                                    heads=3,concat = False,dropout=GAT_dropout,
                                    add_self_loops= True,edge_dim=1)

        self.transformer_layers = nn.ModuleList([
            InterpretableTransformerLayer(
                dim_ff = 512,
                d_model = pathway_hidden_dim1,
                nhead = transformer_heads,
                dropout = 0.3
            ) for _ in range(transformer_layers)
        ])
        # ----------------------
        # 预测头（MLP）
        # ----------------------
        self.net = TwoLForwardNetwork(MLP_input_dim,
                                      MLP_hidden_dim1,
                                      MLP_hidden_dim2,
                                      dropout_pathway,num_class,
                                      activation_func = nn.GELU())
        
        self.activation = nn.GELU()
        self.gene_norm1 = GraphNorm(gene_hidden_dim1)
        self.gene_norm2 = GraphNorm(gene_hidden_dim2)
        #self.pathway_norm1 = GraphNorm(pathway_hidden_dim1)
        self.pre_transformer_proj = nn.Linear(pathway_hidden_dim1, pathway_hidden_dim1)
        self.pre_transformer_ln = nn.LayerNorm(pathway_hidden_dim1)
        self.token_dim_reduction = nn.Linear(gene_hidden_dim1+gene_hidden_dim2,final_pathway_dim)
        
    def forward(self,
            expr_df: pd.DataFrame,
            sample_names: list,
            pathway_edge_index1: torch.Tensor,
            pathway_edge_index2: torch.Tensor,
            pathway_edge_weight1: torch.Tensor,
            pathway_edge_weight2: torch.Tensor,
            batch_size: int) -> torch.Tensor:


        batch_res = processor.process_batch(expr_df, sample_names).cuda()

        gene_hidden1 = self.GCN1(batch_res.x, batch_res.edge_index)
        gene_hidden1 = self.gene_norm1(gene_hidden1, batch_res.batch)
        gene_hidden1 = self.activation(gene_hidden1)
        pathway_feats1,gene_atte1 = self.gene_pool1(gene_hidden1,batch_res.batch)
        #pathway_feats1 = global_mean_pool(gene_hidden1, batch_res.batch) 
        
        gene_hidden2 = self.GCN2(gene_hidden1, batch_res.edge_index)
        gene_hidden2 = self.gene_norm2(gene_hidden2, batch_res.batch)
        gene_hidden2 = self.activation(gene_hidden2)
        #pathway_feats2 = global_mean_pool(gene_hidden2, batch_res.batch)  
        pathway_feats2,gene_atte2 = self.gene_pool2(gene_hidden2,batch_res.batch)
        
        
        pathway_feats = torch.cat([pathway_feats1,pathway_feats2],dim=1)
        
        num_samples = len(sample_names)
        
        if num_samples == batch_size: 
            edge_index_b = pathway_edge_index1
            edge_weight_b = pathway_edge_weight1  
        else:       
            edge_index_b = pathway_edge_index2 
            edge_weight_b = pathway_edge_weight2 
            
        #pathway_batch = torch.arange(num_samples).repeat_interleave(258).cuda()
        
        p_emb1, (edge_idx_attn, gat_attn) = self.pathway_GAT1(
            pathway_feats,
            edge_index_b,
            edge_weight_b,
            return_attention_weights=True
        )
        # p_emb1 = self.pathway_norm1(p_emb1,pathway_batch)
        p_emb1 = self.activation(p_emb1)

        # 残差连接
        x = p_emb1 + pathway_feats
        x = x.view(num_samples, 258, -1)
        x = self.pre_transformer_ln(x)
        x = self.pre_transformer_proj(x)
        x_input = x
        transformer_attn_maps = []
        for layer in self.transformer_layers:
            x, attn = layer(x)
            transformer_attn_maps.append(attn)
        
        #p_emb1 = p_emb1.view(num_samples, 258, -1)
        final_feat = x_input+x
        #final_feat = x + p_emb1
        final_feat = self.token_dim_reduction(final_feat)
        
        # flatten 成样本向量
        final_feat = torch.flatten(final_feat, start_dim=1)
        out = self.net(final_feat)  #  [batch_size, num_class]

        return out,final_feat,gat_attn,edge_idx_attn,transformer_attn_maps,gene_atte1,gene_atte2
    


In [3]:
# ====== Argument Parsing ====== #
parser = argparse.ArgumentParser()
parser.add_argument('--WORKDIR_PATH', type=str, default="/home/nanyuan/hdd/pathway_chat")
parser.add_argument('--model_name', type=str, default='my')
parser.add_argument('--outdir',type=str,default="/home/nanyuan/hdd/pathway_chat/res/test_temp")

# === Train setting === #
parser.add_argument('--learning_rate', type=float, default=1e-4)
parser.add_argument('--epochs', type=int, default=100)
parser.add_argument('--batch_size', type=int, default=64)
parser.add_argument('--weight_decay', type=float, default=0)
parser.add_argument('--patience', type=int, default=10)
parser.add_argument('--testset_yes', type=bool, default=True)
args = parser.parse_known_args()[0]

In [4]:
'''
exp_file  = '/home/nanyuan/hdd/pathway_chat/data/TCGA/cancer_integrate/primary_exp.csv'
exp_data = pd.read_csv(exp_file, index_col=0)
exp_data.to_pickle('/home/nanyuan/hdd/pathway_chat/data/TCGA/cancer_integrate/primary_exp.pkl')
'''
with open("/home/nanyuan/hdd/pathway_chat/data/pathways_adjacency(20).pkl", "rb") as f:  # 注意要用 "rb" 读二进制
    pathways_matrix = pickle.load(f)
pathway_adj = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/pathway_weight_adj(20).csv',index_col=0) 
exp_data = pd.read_pickle('/home/nanyuan/hdd/pathway_chat/data/TCGA/cancer_integrate1/primary_exp.pkl')
label_data = pd.read_csv("/home/nanyuan/hdd/pathway_chat/data/TCGA/cancer_integrate1/primary_label.csv")
pathway_order = list(pathways_matrix.keys())
pathway_adj = pathway_adj.loc[pathway_order, :]
pathway_adj = pathway_adj.loc[:, pathway_order]
#pathway_adj = pathway_adj.where(pathway_adj>= 0.5,0)
sparseTensor = torch.tensor(pathway_adj.values).to_sparse().cuda()
pathway_ind = sparseTensor.indices()
pathway_weight = sparseTensor.values()

'''
# 第一次划分：train vs (val+test)
train_label, val_label = train_test_split(
    label_data,
    test_size=0.2,           # 留20%作 val
    stratify=label_data['label'],
    random_state=8655
)


#val_label['label'].value_counts()
train_exp = exp_data[train_label['samples'].tolist()]
val_exp   = exp_data[val_label['samples'].tolist()]
train_label_dict = dict(zip(train_label['samples'], train_label['label']))
val_label_dict   = dict(zip(val_label['samples'], val_label['label']))
train_dataset = ExpressionDataset(train_exp, train_label_dict)
val_dataset = ExpressionDataset(val_exp, val_label_dict)
train_dataloader = DataLoader(train_dataset,batch_size=args.batch_size,shuffle=True,collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)

edge_index_b = []
for i in range(args.batch_size): 
    offset = i * 258
    edge_index_b.append(pathway_ind + offset) 
            
edge_index_b = torch.cat(edge_index_b, dim=1) 
edge_weight_b = pathway_weight.repeat(args.batch_size)
edge_weight_b = edge_weight_b.float()
      
other_train = train_exp.shape[1] % args.batch_size
other_val = val_exp.shape[1] % args.batch_size
 
edge_index_b_train = []
for i in range(other_train): 
    offset = i * 258
    edge_index_b_train.append(pathway_ind + offset) 
            
edge_index_b_train = torch.cat(edge_index_b_train, dim=1) 
edge_weight_b_train = pathway_weight.repeat(other_train)
edge_weight_b_train = edge_weight_b_train.float()

edge_index_b_val = []
for i in range(other_val): 
    offset = i * 258 
    edge_index_b_val.append(pathway_ind + offset) 
            
edge_index_b_val = torch.cat(edge_index_b_val, dim=1) 
edge_weight_b_val = pathway_weight.repeat(other_val)
edge_weight_b_val = edge_weight_b_val.float()
'''
processor = GenePathwayProcessor(pathways_matrix)

'''
#10%数据测试----
label_data = label_data.groupby("label").apply(
    lambda x: x.sample(frac=0.2)  # frac=0.1 就是抽取10%
).reset_index(drop=True)

sample_names = label_data["samples"].tolist()  # 获取抽取的样本名列表
exp_data = exp_data[sample_names]  # 按样本名提取数据

'''

'\n#10%数据测试----\nlabel_data = label_data.groupby("label").apply(\n    lambda x: x.sample(frac=0.2)  # frac=0.1 就是抽取10%\n).reset_index(drop=True)\n\nsample_names = label_data["samples"].tolist()  # 获取抽取的样本名列表\nexp_data = exp_data[sample_names]  # 按样本名提取数据\n\n'

In [5]:
# train function
def train(model, train_exp, train_loader, edge_index_b,edge_index_b_train, edge_weight_b,edge_weight_b_train,batchsize,loss_fn, optimizer):

    model.train()
    list_train_loss = []
    list_train_out = []
    list_train_true = []
    list_train_samples = []

    optimizer.zero_grad()

    for batch in tqdm(train_loader):
        sample_names = batch['sample_names']  # list[str]
        train_label = batch['labels'].cuda()

        sample_pred,_,_,_,_,_,_= model(train_exp, sample_names, edge_index_b,edge_index_b_train, edge_weight_b.unsqueeze(1),edge_weight_b_train.unsqueeze(1),batchsize)

        #y_true = torch.tensor(train_label[sample_name]).long().cuda()
        loss = loss_fn(sample_pred, train_label)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
            
        list_train_samples.append(sample_names)
        list_train_out.append(sample_pred.detach().cpu().numpy())
        list_train_loss.append(loss.detach().cpu().numpy())
        list_train_true.append(train_label.detach().cpu().numpy())

    return model, list_train_samples, list_train_out,list_train_loss, list_train_true

def validate(model,val_exp, val_loader, edge_index_b,edge_index_b_val, edge_weight_b,edge_weight_b_val,batchsize,loss_fn):
    model.eval()
    with torch.no_grad():  #####禁用梯度计算
    # ====== Test ====== #
        list_val_loss  = []
        list_val_out = []
        list_val_true = []
        list_val_sample = []
        
        for batch in tqdm(val_loader):
            sample_names = batch['sample_names']  # list[str]
            val_label = batch['labels'].cuda()
            
            sample_pred,_,_,_,_,_,_ = model(val_exp, sample_names, edge_index_b,edge_index_b_val, edge_weight_b.unsqueeze(1),edge_weight_b_val.unsqueeze(1),batchsize)
            output_loss = loss_fn(sample_pred, val_label)
            list_val_sample.append(sample_names)
            list_val_out.append(sample_pred.detach().cpu().numpy())
            list_val_loss.append(output_loss.detach().cpu().numpy())
            list_val_true.append(val_label.detach().cpu().numpy())
            
    return  list_val_sample, list_val_out,list_val_loss ,list_val_true

def test(model,test_exp, test_loader, edge_index_b,edge_index_b_test, edge_weight_b,edge_weight_b_test,batchsize,loss_fn):
    model.eval() 
    with torch.no_grad():  #####禁用梯度计算
        # ====== Test ====== #
        list_test_loss  = []
        list_test_out = []
        list_test_true = []
        list_test_sample = []

        for batch in tqdm(test_loader):
            sample_names = batch['sample_names']  # list[str]
            test_label = batch['labels'].cuda()
            
            sample_pred,_,_,_,_,_,_ = model(test_exp, sample_names, edge_index_b,edge_index_b_test, edge_weight_b.unsqueeze(1),edge_weight_b_test.unsqueeze(1),batchsize)
            output_loss = loss_fn(sample_pred, test_label)

            list_test_sample.append(sample_names)
            list_test_out.append(sample_pred.detach().cpu().numpy())
            list_test_loss.append(output_loss.detach().cpu().numpy())
            list_test_true.append(test_label.detach().cpu().numpy())
            
    return  list_test_sample,list_test_out,list_test_loss,list_test_true


In [6]:
def experiment(args,train_exp,val_exp,train_loader,val_loader,
               edge_index_b,edge_index_b_train,edge_index_b_val,
               edge_weight_b,edge_weight_b_train,edge_weight_b_val,batchsize,model,loss_fn): 
    
    # === Optimizer === #
    
    optimizer = optim.Adam([
            {'params': model.parameters()},
            {'params': loss_fn.parameters()}
        ], lr=args.learning_rate, weight_decay=args.weight_decay) 
  
    # === Scheduler === #
    #监控训练过程，自动调整学习率等参数
    #lf = lambda x: ((1 + math.cos(x * math.pi / args.epochs)) / 2) * (1 - 1e-3) + 1e-3  # cosine
  
    # === Scheduler === #
    #监控训练过程，自动调整学习率等参数
    #scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=4)
    

    # ====== Cross Validation Best Performance Dict ====== #
    best_performances = {}
    best_performances['best_epoch'] = 0
    best_performances['best_train_loss'] = float('inf')
    best_performances['best_train_AUPRC'] = 0.0
    best_performances['best_train_ACC'] = 0.0
    best_performances['best_train_F1'] = 0.0
    best_performances['best_train_Precision'] = 0.0
    best_performances['best_train_Recall'] = 0.0
    best_performances['best_valid_loss'] = float('inf')
    best_performances['best_valid_AUPRC'] = 0.0
    best_performances['best_valid_ACC'] = 0.0
    best_performances['best_valid_F1'] = 0.0
    best_performances['best_valid_Precision'] = 0.0
    best_performances['best_valid_Recall'] = 0.0
    # ==================================================== #
    list_epoch = []
  
    list_train_epoch_loss,list_epoch_AUPRC,list_epoch_ACC,list_epoch_F1,list_epoch_Precision,list_epoch_Recall = [],[],[],[],[],[]

    
    list_val_epoch_loss,list_val_epoch_AUPRC,list_val_epoch_ACC,list_val_epoch_F1,list_val_epoch_Precision,list_val_epoch_Recall = [],[],[],[],[],[]


    counter = 0
    #epoch = 0
    for epoch in range(args.epochs):
        list_epoch.append(epoch)
        # ====== TRAIN Epoch ====== #
        ### 训练模型，并获取训练过程中的损失和输出结果
        model, _, list_train_out,list_train_loss, list_train_true = train(
            model, train_exp, train_loader, edge_index_b,edge_index_b_train, edge_weight_b,edge_weight_b_train,batchsize,loss_fn, optimizer)

        logits_pred = np.vstack(list_train_out)
        y_true = np.concatenate(list_train_true)
        logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
        probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
        epoch_train_AUC = roc_auc_score(y_true, probs, average="weighted", multi_class="ovr")
        epoch_train_AUPRC = average_precision_score(y_true, probs, average="weighted")
        y_pred = np.argmax(probs, axis=1)
        epoch_train_ACC = accuracy_score(y_true, y_pred)
        epoch_train_MCC = matthews_corrcoef(y_true, y_pred)
        epoch_train_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        epoch_train_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        epoch_train_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        train_epoch_loss = sum(list_train_loss) / len(list_train_loss)
        # 将训练过程中的指标记录下来
        list_train_epoch_loss.append(train_epoch_loss)
        list_epoch_AUPRC.append(epoch_train_AUPRC)
        list_epoch_ACC.append(epoch_train_ACC)
        list_epoch_F1.append(epoch_train_F1)
        list_epoch_Precision.append(epoch_train_Precision)
        list_epoch_Recall.append(epoch_train_Recall)
        # ====== VALID Epoch ====== #
        
        # 在验证集上评估模型，并获取验证过程中的损失和输出结果
        
        val_samples,list_val_out,list_val_loss ,list_val_true = validate(
            model,val_exp, val_loader, edge_index_b,edge_index_b_val, edge_weight_b,edge_weight_b_val,batchsize,loss_fn)
        logits_pred = np.vstack(list_val_out)
        y_true = np.concatenate(list_val_true)
        logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
        probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
        epoch_val_AUC = roc_auc_score(y_true, probs, average="weighted", multi_class="ovr")
        epoch_val_AUPRC = average_precision_score(y_true, probs, average="weighted")
        y_pred = np.argmax(probs, axis=1)
        epoch_val_ACC = accuracy_score(y_true, y_pred)
        epoch_val_MCC = matthews_corrcoef(y_true, y_pred)
        epoch_val_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        epoch_val_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        epoch_val_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        val_epoch_loss = sum(list_val_loss)/len(list_val_loss)

        list_val_epoch_loss.append(val_epoch_loss)
        list_val_epoch_AUPRC.append(epoch_val_AUPRC)
        list_val_epoch_ACC.append(epoch_val_ACC)
        list_val_epoch_F1.append(epoch_val_F1)
        list_val_epoch_Precision.append(epoch_val_Precision)
        list_val_epoch_Recall.append(epoch_val_Recall)

        # 检查当前模型在验证集上的性能是否优于之前记录的最佳性能
        if val_epoch_loss < best_performances['best_valid_loss']:
            # 如果是，更新最佳性能记录，并保存当前模型
            best_performances['best_epoch'] = epoch
            best_performances['best_train_loss'] = train_epoch_loss.item()
            best_performances['best_train_AUPRC'] = epoch_train_AUPRC.item()
            best_performances['best_train_ACC'] = epoch_train_ACC
            best_performances['best_train_F1'] = epoch_train_F1
            best_performances['best_train_Precision'] = epoch_train_Precision
            best_performances['best_train_Recall'] = epoch_train_Recall
            best_performances['best_valid_loss'] = val_epoch_loss.item()
            best_performances['best_valid_AUPRC'] = epoch_val_AUPRC.item()
            best_performances['best_valid_ACC'] = epoch_val_ACC
            best_performances['best_valid_F1'] = epoch_val_F1
            best_performances['best_valid_Precision'] = epoch_val_Precision
            best_performances['best_valid_Recall'] = epoch_val_Recall

            list_train_pred = list_train_out
            list_val_pred = list_val_out
            list_val_samples = val_samples
            #torch.save(model, os.path.join(args.outdir + '/train.model'))
            model_max = deepcopy(model) # 深拷贝当前模型，以保存最佳模型的状态
            # 重置计数器
            counter = 0
        else:
            # 如果验证集上的性能没有提升（说明前一个是最好的），增加计数器
            counter += 1
            logging(f'Early Stopping counter: {counter} out of {args.patience}', args.outdir, 'stop_count.log')
        # 记录当前 epoch 的性能信息              
        logging(f'Epoch: {epoch:02d}, Train:\t loss: {list_train_epoch_loss[-1]:.4f},AUC: {epoch_train_AUC:.4f}, PRC: {epoch_train_AUPRC:.4f},ACC: {epoch_train_ACC:.4f},\
                MCC: {epoch_train_MCC:.4f},Precision: {epoch_train_Precision:.4f},Recall: {epoch_train_Recall:.4f},F1: {epoch_train_F1:.4f}\
                    val:loss: {val_epoch_loss:.4f},auc: {epoch_val_AUC:.4f},PRC: {epoch_val_AUPRC:.4f},ACC: {epoch_val_ACC:.4f},MCC: {epoch_val_MCC:.4f},\
                        Precision: {epoch_val_Precision:.4f},Recall: {epoch_val_Recall:.4f},F1: {epoch_val_F1:.4f}', args.outdir, 'train_record.log')
                   
        # 如果计数器达到指定的耐心值，提前停止训练
        if counter == args.patience:
            break
        # 根据验证集上的损失动态调整学习率,当前设置patience=4，忍耐四次，第五次调整学习率
        scheduler.step(list_val_epoch_loss[-1])
        #print(list_val_epoch_loss[-1],type(list_val_epoch_loss[-1]))
        #logging(f'LR after update:{optimizer.param_groups[0]["lr"]}',args.outdir,'lr.log')
        log_learning_rates(optimizer, args.outdir)
    
    # 将训练、验证和测试集上的性能指标和最佳模型的信息保存到字典中
    result = {}
    result['train_losses'] = list(map(lambda x: round(float(x), 6), list_train_epoch_loss))
    result['val_losses'] = list(map(lambda x: round(float(x), 6), list_val_epoch_loss))
    result['train_AUPRC'] = list(map(lambda x: round(float(x), 6), list_epoch_AUPRC))
    result['val_AUPRC'] = list(map(lambda x: round(float(x), 6), list_val_epoch_AUPRC))
    # 将最佳性能信息保存到文件中
    filename = os.path.join(args.outdir, 'model_best_performances.json')
    with open(filename, 'w') as f:
        json.dump(best_performances, f)
    # 返回参数、训练结果、最佳性能信息和最佳模型
    return vars(args), result, best_performances,list_train_pred,list_val_pred,model_max,list_val_samples

In [ ]:
seed_everything(8655)
if __name__ == '__main__':
    # 五折交叉验证
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=8655)

    samples = label_data['samples'].values
    labels = label_data['label'].values

    fold_id = 1
    total_results = defaultdict(list)

    for train_index, val_index in skf.split(samples, labels):

        print(f"======== Fold {fold_id} / 5 ========")

        train_samples = samples[train_index]
        val_samples   = samples[val_index]

        train_exp = exp_data[train_samples]
        val_exp   = exp_data[val_samples]

        train_label_dict = dict(zip(train_samples, labels[train_index]))
        val_label_dict   = dict(zip(val_samples, labels[val_index]))

        train_dataset = ExpressionDataset(train_exp, train_label_dict)
        val_dataset   = ExpressionDataset(val_exp, val_label_dict)

        train_dataloader = DataLoader(
            train_dataset,
            batch_size=args.batch_size,
            shuffle=True,
            collate_fn=collate_fn
        )

        val_dataloader = DataLoader(
            val_dataset,
            batch_size=args.batch_size,
            shuffle=False,
            collate_fn=collate_fn
        )

        # 2. 构建 edge index（与你原来完全一致）
        edge_index_b = []
        for i in range(args.batch_size):
            offset = i * 258
            edge_index_b.append(pathway_ind + offset)
        edge_index_b = torch.cat(edge_index_b, dim=1)
        edge_weight_b = pathway_weight.repeat(args.batch_size).float()

        other_train = train_exp.shape[1] % args.batch_size
        other_val   = val_exp.shape[1] % args.batch_size

        edge_index_b_train = []
        for i in range(other_train):
            offset = i * 258
            edge_index_b_train.append(pathway_ind + offset)
        edge_index_b_train = torch.cat(edge_index_b_train, dim=1)
        edge_weight_b_train = pathway_weight.repeat(other_train).float()

        edge_index_b_val = []
        for i in range(other_val):
            offset = i * 258
            edge_index_b_val.append(pathway_ind + offset)
        edge_index_b_val = torch.cat(edge_index_b_val, dim=1)
        edge_weight_b_val = pathway_weight.repeat(other_val).float()

        # 3. 初始化模型与 loss
        model = PathwayGATModel(
            gene_in_dim=1, gene_hidden_dim1=32, gene_hidden_dim2=96,
            pathway_in_dim=128, pathway_hidden_dim1=128,
            MLP_input_dim=8256 , MLP_hidden_dim1=512, MLP_hidden_dim2=128,
            num_class=25, dropout_pathway=0.5, GAT_dropout=0.3,transformer_heads = 4,
            transformer_layers = 3,final_pathway_dim=32).cuda()

        # class weights
        counts = torch.tensor(label_data['label'].value_counts().sort_index().values, dtype=torch.float).cuda()

        n_classes = len(counts)
        total = counts.sum()
        class_weights = total / (n_classes * counts).cuda()

        loss_fn = nn.CrossEntropyLoss(weight=class_weights)

        # 4. 运行 experiment
        setting, result, best_performances, train_pred, val_pred,model_max,val_samples = experiment(
            args, train_exp, val_exp, train_dataloader, val_dataloader,
            edge_index_b, edge_index_b_train, edge_index_b_val,
            edge_weight_b, edge_weight_b_train, edge_weight_b_val,
            args.batch_size, model, loss_fn
        )
        val_samples = np.concatenate(val_samples)
        # 5. 记录本折结果
        total_results['fold'].append(fold_id)
        total_results['best_epoch'].append(best_performances['best_epoch'])
        total_results['best_train_loss'].append(best_performances['best_train_loss'])
        total_results['best_train_AUPRC'].append(best_performances['best_train_AUPRC'])
        total_results['best_train_ACC'].append(best_performances['best_train_ACC'])
        total_results['best_train_F1'].append(best_performances['best_train_F1'])
        total_results['best_train_Precision'].append(best_performances['best_train_Precision'])
        total_results['best_train_Recall'].append(best_performances['best_train_Recall'])
        total_results['best_valid_loss'].append(best_performances['best_valid_loss'])
        total_results['best_valid_AUPRC'].append(best_performances['best_valid_AUPRC'])
        total_results['best_valid_ACC'].append(best_performances['best_valid_ACC'])
        total_results['best_valid_F1'].append(best_performances['best_valid_F1'])
        total_results['best_valid_Precision'].append(best_performances['best_valid_Precision'])
        total_results['best_valid_Recall'].append(best_performances['best_valid_Recall'])
        

        # 可选：为每折单独保存模型
        torch.save(model_max, os.path.join(args.outdir, f'fold_{fold_id}_best.model'))
        np.save(os.path.join(args.outdir, f'fold_{fold_id}_val_samples'), val_samples)
        fold_id += 1

    print("======== Cross Validation Summary ========")
    print(total_results)

In [8]:
pd.DataFrame(total_results).to_csv(os.path.join(args.outdir, "cv_summary.csv"), index=False)

TEST

In [9]:
args.outdir

'/home/nanyuan/hdd/pathway_chat/res/test_temp'

In [10]:
pd.read_csv('/home/nanyuan/hdd/pathway_chat/res/test_temp/cv_summary.csv')

,fold,best_epoch,best_train_loss,best_train_AUPRC,best_train_ACC,best_train_F1,best_train_Precision,best_train_Recall,best_valid_loss,best_valid_AUPRC,best_valid_ACC,best_valid_F1,best_valid_Precision,best_valid_Recall
0,1,29,0.032642,0.999949,0.997674,0.997674,0.997680,0.997674,0.122247,0.991742,0.975930,0.975931,0.976493,0.975930
1,2,26,0.037167,0.999952,0.997811,0.997813,0.997821,0.997811,0.120598,0.992810,0.977024,0.977356,0.978383,0.977024
2,3,31,0.026146,0.999932,0.998222,0.998221,0.998223,0.998222,0.116635,0.991769,0.975930,0.976140,0.976730,0.975930
3,4,37,0.021185,0.999980,0.999043,0.999047,0.999061,0.999043,0.121201,0.990497,0.974275,0.974344,0.974758,0.974275
4,5,43,0.028041,0.999985,0.998906,0.998907,0.998913,0.998906,0.128473,0.990853,0.973180,0.973441,0.974262,0.973180


In [11]:
model_max = torch.load('/home/nanyuan/hdd/pathway_chat/res/test_temp/fold_2_best.model',weights_only=False)
counts = torch.tensor(label_data['label'].value_counts().sort_index().values, dtype=torch.float).cuda()
n_classes = len(counts)
total = counts.sum()
class_weights = total / (n_classes * counts).cuda()
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

In [13]:
edge_index_b = []
for i in range(args.batch_size):
    offset = i * 258
    edge_index_b.append(pathway_ind + offset)
edge_index_b = torch.cat(edge_index_b, dim=1)
edge_weight_b = pathway_weight.repeat(args.batch_size).float()

label_dict  = dict(zip(label_data['samples'], label_data['label']))
all_dataset = ExpressionDataset(exp_data, label_dict)
all_dataloader = DataLoader(all_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_all = exp_data.shape[1] % args.batch_size
edge_index_b_all = []
for i in range(other_all): 
    offset = i * 258 
    edge_index_b_all.append(pathway_ind + offset) 

edge_index_b_all = torch.cat(edge_index_b_all, dim=1) 
edge_weight_b_all = pathway_weight.repeat(other_all)
edge_weight_b_all = edge_weight_b_all.float()

for batch in tqdm(all_dataloader):
    sample_names = batch['sample_names']  # list[str]
    val_label = batch['labels'].cuda()
    
sample_pred,_,gat_atten,_,transformer_attn_maps,_,_ = model_max(exp_data, sample_names, edge_index_b,edge_index_b_all, edge_weight_b.unsqueeze(1),edge_weight_b_all.unsqueeze(1),args.batch_size)
attn_stacked = torch.stack(transformer_attn_maps, dim=0)
final_attn_matrix = attn_stacked.mean(dim=[0, 1, 2])
sorted_values, sorted_indices  = torch.sort(final_attn_matrix.mean(dim=0),descending=True)
print(sorted_values)
print(sorted_values.max())
print(sorted_values.min())
#sorted_values


  0%|          | 0/143 [00:00<?, ?it/s]

100%|██████████| 143/143 [00:00<00:00, 2475.72it/s]


tensor([0.0076, 0.0073, 0.0072, 0.0066, 0.0064, 0.0063, 0.0062, 0.0059, 0.0058,
        0.0057, 0.0057, 0.0056, 0.0056, 0.0056, 0.0056, 0.0055, 0.0055, 0.0055,
        0.0053, 0.0053, 0.0053, 0.0053, 0.0053, 0.0052, 0.0052, 0.0052, 0.0052,
        0.0050, 0.0050, 0.0049, 0.0049, 0.0049, 0.0048, 0.0048, 0.0048, 0.0047,
        0.0047, 0.0047, 0.0047, 0.0047, 0.0046, 0.0046, 0.0046, 0.0046, 0.0046,
        0.0046, 0.0045, 0.0045, 0.0045, 0.0045, 0.0045, 0.0045, 0.0044, 0.0044,
        0.0044, 0.0044, 0.0044, 0.0043, 0.0043, 0.0043, 0.0043, 0.0043, 0.0043,
        0.0043, 0.0043, 0.0043, 0.0043, 0.0043, 0.0043, 0.0043, 0.0043, 0.0043,
        0.0043, 0.0043, 0.0043, 0.0042, 0.0042, 0.0042, 0.0042, 0.0042, 0.0042,
        0.0042, 0.0042, 0.0042, 0.0042, 0.0041, 0.0041, 0.0041, 0.0041, 0.0040,
        0.0040, 0.0040, 0.0040, 0.0039, 0.0039, 0.0039, 0.0039, 0.0039, 0.0039,
        0.0039, 0.0039, 0.0039, 0.0039, 0.0039, 0.0039, 0.0039, 0.0039, 0.0039,
        0.0039, 0.0038, 0.0038, 0.0038, 

In [14]:
val_samples = np.load("/home/nanyuan/hdd/pathway_chat/res/test_temp/fold_2_val_samples.npy")
val_exp = exp_data[val_samples]
val_label = label_data[label_data['samples'].isin(val_samples)]
label_dict  = dict(zip(val_label['samples'], val_label['label']))
val_dataset = ExpressionDataset(val_exp, label_dict)
val_dataloader = DataLoader(val_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_val = val_exp.shape[1] % args.batch_size
edge_index_b_val = []
for i in range(other_val): 
    offset = i * 258 
    edge_index_b_val.append(pathway_ind + offset) 

edge_index_b_val = torch.cat(edge_index_b_val, dim=1) 
edge_weight_b_val = pathway_weight.repeat(other_val)
edge_weight_b_val = edge_weight_b_val.float()


_,list_val_out,list_val_loss,list_val_true= test(
    model_max,val_exp, val_dataloader, edge_index_b,edge_index_b_val, edge_weight_b,edge_weight_b_val,args.batch_size,loss_fn)

logits_pred = np.vstack(list_val_out)
y_true = np.concatenate(list_val_true)
logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
y_true_onehot = np.zeros((len(y_true), 25))
for i, lab in enumerate(y_true):
    y_true_onehot[i, int(lab)] = 1
test_AUPRC = average_precision_score(y_true_onehot, probs, average="weighted")
y_pred = np.argmax(probs, axis=1)
test_ACC = accuracy_score(y_true, y_pred)
test_MCC = matthews_corrcoef(y_true, y_pred)
test_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
test_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
test_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
test_loss = sum(list_val_loss)/len(list_val_loss)

print('test_AUPR',test_AUPRC)
print('test_ACC:',test_ACC)
print('test_Precision:',test_Precision)
print('test_Recall:',test_Recall)
print('test_F1:',test_F1)
print('test_MCC',test_MCC)
print('test_loss:',test_loss)


100%|██████████| 29/29 [00:45<00:00,  1.57s/it]

test_AUPR 0.9928101480897666
test_ACC: 0.9770240700218819
test_Precision: 0.9783829986754994
test_Recall: 0.9770240700218819
test_F1: 0.9773564629427617
test_MCC 0.9754439783539256
test_loss: 0.120597996


In [ ]:
val_samples = val_exp.columns.to_numpy()
final_lable_data = np.column_stack([y_true, y_pred, probs])
final_lable_data = pd.DataFrame(
    final_lable_data,
    columns=["True_label", "Pre_label"] + [f"Class{i}_Prob" for i in range(25)],  # 列名
    index=val_samples  # 行名（样本名称）
)
final_lable_data.to_csv('/home/nanyuan/hdd/pathway_chat/method_compare/val_label/ourmethod/TCGA_primary_label.csv')

In [15]:
#转移样本
meta_label_data = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/TCGA/cancer_integrate1/meta_label.csv')
meta_exp_file  = '/home/nanyuan/hdd/pathway_chat/data/TCGA/cancer_integrate1/meta_exp.csv'
meta_exp_data = pd.read_csv(meta_exp_file, index_col=0)
meta_label_dict  = dict(zip(meta_label_data['samples'], meta_label_data['label']))
meta_dataset = ExpressionDataset(meta_exp_data, meta_label_dict)
meta_dataloader = DataLoader(meta_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_meta = meta_exp_data.shape[1] % args.batch_size
edge_index_b_meta = []
for i in range(other_meta): 
    offset = i * 258 
    edge_index_b_meta.append(pathway_ind + offset) 
            
edge_index_b_meta = torch.cat(edge_index_b_meta, dim=1) 
edge_weight_b_meta = pathway_weight.repeat(other_meta)
edge_weight_b_meta = edge_weight_b_meta.float()


_,list_test_out,list_test_loss,list_test_true = test(
    model_max,meta_exp_data, meta_dataloader, edge_index_b,edge_index_b_meta, edge_weight_b,edge_weight_b_meta,args.batch_size,loss_fn)

logits_pred = np.vstack(list_test_out)
y_true = np.concatenate(list_test_true)
logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
#test_AUC = roc_auc_score(y_true, probs, average="weighted", multi_class="ovr")
y_true_onehot = np.zeros((len(y_true), 25))
for i, lab in enumerate(y_true):
    y_true_onehot[i, int(lab)] = 1
test_AUPRC = average_precision_score(y_true_onehot, probs, average="weighted")
y_pred = np.argmax(probs, axis=1)
test_ACC = accuracy_score(y_true, y_pred)
test_MCC = matthews_corrcoef(y_true, y_pred)
test_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
test_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
test_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
test_loss = sum(list_test_loss)/len(list_test_loss)

print('test_AUPR',test_AUPRC)
print('test_ACC:',test_ACC)
print('test_Precision:',test_Precision)
print('test_Recall:',test_Recall)
print('test_F1:',test_F1)
print('test_MCC',test_MCC)
print('test_loss:',test_loss)



100%|██████████| 7/7 [00:09<00:00,  1.37s/it]


test_AUPR 0.9893245537523337
test_ACC: 0.9107142857142857
test_Precision: 0.9948979591836735
test_Recall: 0.9107142857142857
test_F1: 0.9503205128205129
test_MCC 0.5986910410267281
test_loss: 0.6480546


In [ ]:
meta_samples = meta_exp_data.columns.to_numpy()
final_lable_data = np.column_stack([y_true, y_pred, probs])
final_lable_data = pd.DataFrame(
    final_lable_data,
    columns=["True_label", "Pre_label"] + [f"Class{i}_Prob" for i in range(25)],  # 列名
    index=meta_samples  # 行名（样本名称）
)
final_lable_data.to_csv('/home/nanyuan/hdd/pathway_chat/method_compare/val_label/ourmethod/TCGA_meta_label.csv')

In [16]:
#ICGC
icgc_label_data = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/ICGC/cancer_integrate1/label_data.csv')
icgc_exp_file  = '/home/nanyuan/hdd/pathway_chat/data/ICGC/cancer_integrate1/exp_data_inte.csv'
icgc_exp_data = pd.read_csv(icgc_exp_file, index_col=0)
icgc_label_dict  = dict(zip(icgc_label_data['samples'], icgc_label_data['label']))
icgc_dataset = ExpressionDataset(icgc_exp_data, icgc_label_dict)
icgc_dataloader = DataLoader(icgc_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_icgc = icgc_exp_data.shape[1] % args.batch_size
edge_index_b_icgc = []
for i in range(other_icgc): 
    offset = i * 258 
    edge_index_b_icgc.append(pathway_ind + offset) 
            
edge_index_b_icgc = torch.cat(edge_index_b_icgc, dim=1) 
edge_weight_b_icgc = pathway_weight.repeat(other_icgc)
edge_weight_b_icgc = edge_weight_b_icgc.float()


_,list_icgc_out,list_icgc_loss,list_icgc_true= test(
    model_max,icgc_exp_data, icgc_dataloader, edge_index_b,edge_index_b_icgc, edge_weight_b,edge_weight_b_icgc,args.batch_size,loss_fn)

logits_pred = np.vstack(list_icgc_out)
y_true = np.concatenate(list_icgc_true)
logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
y_true_onehot = np.zeros((len(y_true), 25))
for i, lab in enumerate(y_true):
    y_true_onehot[i, int(lab)] = 1
icgc_AUPRC = average_precision_score(y_true_onehot, probs, average="weighted")
y_pred = np.argmax(probs, axis=1)
icgc_ACC = accuracy_score(y_true, y_pred)
icgc_MCC = matthews_corrcoef(y_true, y_pred)
icgc_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
icgc_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
icgc_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
icgc_loss = sum(list_icgc_loss)/len(list_icgc_loss)

print('icgc_AUPRC:',icgc_AUPRC)
print('icgc_ACC:',icgc_ACC)
print('icgc_Precision:',icgc_Precision)
print('icgc_Recall:',icgc_Recall)
print('icgc_F1:',icgc_F1)
print('icgc_MCC:',icgc_MCC)
print('icgc_loss:',icgc_loss)


100%|██████████| 118/118 [03:17<00:00,  1.68s/it]

icgc_AUPRC: 0.9982895758958311
icgc_ACC: 0.9882854100106496
icgc_Precision: 0.991230960151882
icgc_Recall: 0.9882854100106496
icgc_F1: 0.9896639900428732
icgc_MCC: 0.9872711730027894
icgc_loss: 0.09777381


In [ ]:
icgc_p_samples = icgc_exp_data.columns.to_numpy()
final_lable_data = np.column_stack([y_true, y_pred, probs])
final_lable_data = pd.DataFrame(
    final_lable_data,
    columns = ["True_label", "Pre_label"] + [f"Class{i}_Prob" for i in range(25)],  # 列名
    index = icgc_p_samples  # 行名（样本名称）
)
final_lable_data.to_csv('/home/nanyuan/hdd/pathway_chat/method_compare/val_label/ourmethod/ICGC_primary_label.csv')

In [17]:
#icgc_meta
icgc_meta_label_data = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/ICGC/cancer_integrate1/label_data_meta.csv')
icgc_meta_exp_file  = '/home/nanyuan/hdd/pathway_chat/data/ICGC/cancer_integrate1/exp_tpm_meta.csv'
icgc_meta_exp_data = pd.read_csv(icgc_meta_exp_file, index_col=0)
icgc_meta_label_dict  = dict(zip(icgc_meta_label_data['samples'], icgc_meta_label_data['label']))
icgc_meta_dataset = ExpressionDataset(icgc_meta_exp_data, icgc_meta_label_dict)
icgc_meta_dataloader = DataLoader(icgc_meta_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_icgc_meta = icgc_meta_exp_data.shape[1] % args.batch_size
edge_index_b_icgc_meta = []
for i in range(other_icgc_meta): 
    offset = i * 258 
    edge_index_b_icgc_meta.append(pathway_ind + offset) 
            
edge_index_b_icgc_meta = torch.cat(edge_index_b_icgc_meta, dim=1) 
edge_weight_b_icgc_meta = pathway_weight.repeat(other_icgc_meta)
edge_weight_b_icgc_meta = edge_weight_b_icgc_meta.float()


_,list_icgc_meta_out,list_icgc_meta_loss,list_icgc_meta_true = test(
    model_max,icgc_meta_exp_data, icgc_meta_dataloader, edge_index_b,edge_index_b_icgc_meta, edge_weight_b,edge_weight_b_icgc_meta,args.batch_size,loss_fn)

logits_pred = np.vstack(list_icgc_meta_out)
y_true = np.concatenate(list_icgc_meta_true)
logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
y_true_onehot = np.zeros((len(y_true), 25))
for i, lab in enumerate(y_true):
    y_true_onehot[i, int(lab)] = 1
icgc_meta_AUPRC = average_precision_score(y_true_onehot, probs, average="weighted")
y_pred = np.argmax(probs, axis=1)
icgc_meta_ACC = accuracy_score(y_true, y_pred)
icgc_meta_MCC = matthews_corrcoef(y_true, y_pred)
icgc_meta_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
icgc_meta_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
icgc_meta_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
icgc_meta_loss = sum(list_icgc_meta_loss)/len(list_icgc_meta_loss)

print('icgc_meta_AUPRC:',icgc_meta_AUPRC)
print('icgc_meta_ACC:',icgc_meta_ACC)
print('icgc_meta_Precision:',icgc_meta_Precision)
print('icgc_meta_Recall:',icgc_meta_Recall)
print('icgc_meta_F1:',icgc_meta_F1)
print('icgc_meta_MCC:',icgc_meta_MCC)
print('icgc_meta_loss:',icgc_meta_loss)


100%|██████████| 6/6 [00:08<00:00,  1.41s/it]

icgc_meta_AUPRC: 0.9934043351767643
icgc_meta_ACC: 0.9276139410187667
icgc_meta_Precision: 0.9950848972296694
icgc_meta_Recall: 0.9276139410187667
icgc_meta_F1: 0.959505794444824
icgc_meta_MCC: 0.6103004249241378
icgc_meta_loss: 0.29682603


In [ ]:
icgc_m_samples = icgc_meta_exp_data.columns.to_numpy()
final_lable_data = np.column_stack([y_true, y_pred, probs])
final_lable_data = pd.DataFrame(
    final_lable_data,
    columns = ["True_label", "Pre_label"] + [f"Class{i}_Prob" for i in range(25)],  # 列名
    index = icgc_m_samples  # 行名（样本名称）
)
final_lable_data.to_csv('/home/nanyuan/hdd/pathway_chat/method_compare/val_label/ourmethod/ICGC_meta_label.csv')

In [18]:
#CPTAC
CPTAC_label_data = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/CPTAC1/cancer_integrate1/label_data.csv')
CPTAC_exp_file  = '/home/nanyuan/hdd/pathway_chat/data/CPTAC1/cancer_integrate1/exp_data_inte.csv'
CPTAC_exp_data = pd.read_csv(CPTAC_exp_file, index_col=0)
CPTAC_label_dict  = dict(zip(CPTAC_label_data['samples'], CPTAC_label_data['label']))
CPTAC_dataset = ExpressionDataset(CPTAC_exp_data, CPTAC_label_dict)
CPTAC_dataloader = DataLoader(CPTAC_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_CPTAC = CPTAC_exp_data.shape[1] % args.batch_size
edge_index_b_CPTAC = []
for i in range(other_CPTAC): 
    offset = i * 258 
    edge_index_b_CPTAC.append(pathway_ind + offset) 
            
edge_index_b_CPTAC = torch.cat(edge_index_b_CPTAC, dim=1) 
edge_weight_b_CPTAC = pathway_weight.repeat(other_CPTAC)
edge_weight_b_CPTAC = edge_weight_b_CPTAC.float()


_,list_CPTAC_out,list_CPTAC_loss,list_CPTAC_true = test(
    model_max,CPTAC_exp_data, CPTAC_dataloader, edge_index_b,edge_index_b_CPTAC, edge_weight_b,edge_weight_b_CPTAC,args.batch_size,loss_fn)

logits_pred = np.vstack(list_CPTAC_out)
y_true = np.concatenate(list_CPTAC_true)
logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
y_true_onehot = np.zeros((len(y_true), 25))
for i, lab in enumerate(y_true):
    y_true_onehot[i, int(lab)] = 1
CPTAC_AUPRC = average_precision_score(y_true_onehot, probs, average="weighted")
y_pred = np.argmax(probs, axis=1)
CPTAC_ACC = accuracy_score(y_true, y_pred)
CPTAC_MCC = matthews_corrcoef(y_true, y_pred)
CPTAC_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
CPTAC_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
CPTAC_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
CPTAC_loss = sum(list_CPTAC_loss)/len(list_CPTAC_loss)

print('CPTAC_AUPRC:',CPTAC_AUPRC)
print('CPTAC_ACC:',CPTAC_ACC)
print('CPTAC_Precision:',CPTAC_Precision)
print('CPTAC_Recall:',CPTAC_Recall)
print('CPTAC_F1:',CPTAC_F1)
print('CPTAC_MCC:',CPTAC_MCC)
print('CPTAC_loss:',CPTAC_loss)


100%|██████████| 27/27 [00:45<00:00,  1.69s/it]

CPTAC_AUPRC: 0.9730708025657119
CPTAC_ACC: 0.7992998833138857
CPTAC_Precision: 0.9870470087330679
CPTAC_Recall: 0.7992998833138857
CPTAC_F1: 0.8768895602955462
CPTAC_MCC: 0.7744757150365079
CPTAC_loss: 0.9096062


In [ ]:
'''
from sklearn.metrics import confusion_matrix

labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
df_cm = pd.DataFrame(cm, index=labels, columns=labels)
df_cm.to_csv('/home/nanyuan/hdd/pathway_chat/res/temp/CPTAC.csv')
'''

In [ ]:
cptac_samples = CPTAC_exp_data.columns.to_numpy()
final_lable_data = np.column_stack([y_true, y_pred, probs])
final_lable_data = pd.DataFrame(
    final_lable_data,
    columns = ["True_label", "Pre_label"] + [f"Class{i}_Prob" for i in range(25)],  # 列名
    index = cptac_samples  # 行名（样本名称）
)
final_lable_data.to_csv('/home/nanyuan/hdd/pathway_chat/method_compare/val_label/ourmethod/CPTAC_label.csv')

In [19]:
#MET500
MET500_label_data = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/MET500/cancer_integrate1/label_data.csv')
MET500_exp_file  = '/home/nanyuan/hdd/pathway_chat/data/MET500/cancer_integrate1/exp_data_inte.csv'
MET500_exp_data = pd.read_csv(MET500_exp_file, index_col=0)
MET500_exp_data.columns = MET500_exp_data.columns.str.replace('.', '-', regex=False)
MET500_label_dict  = dict(zip(MET500_label_data['samples'], MET500_label_data['label']))
MET500_dataset = ExpressionDataset(MET500_exp_data, MET500_label_dict)
MET500_dataloader = DataLoader(MET500_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_MET500 = MET500_exp_data.shape[1] % args.batch_size
edge_index_b_MET500 = []
for i in range(other_MET500): 
    offset = i * 258 
    edge_index_b_MET500.append(pathway_ind + offset) 
            
edge_index_b_MET500 = torch.cat(edge_index_b_MET500, dim=1) 
edge_weight_b_MET500 = pathway_weight.repeat(other_MET500)
edge_weight_b_MET500 = edge_weight_b_MET500.float()


_,list_MET500_out,list_MET500_loss,list_MET500_true = test(
    model_max,MET500_exp_data, MET500_dataloader, edge_index_b,edge_index_b_MET500, edge_weight_b,edge_weight_b_MET500,args.batch_size,loss_fn)

logits_pred = np.vstack(list_MET500_out)
y_true = np.concatenate(list_MET500_true)
logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
y_true_onehot = np.zeros((len(y_true), 25))
for i, lab in enumerate(y_true):
    y_true_onehot[i, int(lab)] = 1
MET500_AUPRC = average_precision_score(y_true_onehot, probs, average="weighted")   
y_pred = np.argmax(probs, axis=1)
MET500_ACC = accuracy_score(y_true, y_pred)
MET500_MCC = matthews_corrcoef(y_true, y_pred)
MET500_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
MET500_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
MET500_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
MET500_loss = sum(list_MET500_loss)/len(list_MET500_loss)

print('MET500_AUPR:',MET500_AUPRC)
print('MET500_ACC:',MET500_ACC)
print('MET500_Precision:',MET500_Precision)
print('MET500_Recall:',MET500_Recall)
print('MET500_F1:',MET500_F1)
print('MET500_MCC:',MET500_MCC)
print('MET500_loss:',MET500_loss)


100%|██████████| 12/12 [00:18<00:00,  1.51s/it]

MET500_AUPR: 0.7488815620689429
MET500_ACC: 0.5584958217270195
MET500_Precision: 0.7591045681738697
MET500_Recall: 0.5584958217270195
MET500_F1: 0.6086298462134346
MET500_MCC: 0.5194067469289362
MET500_loss: 1.753162


In [ ]:
MET500_samples = MET500_exp_data.columns.to_numpy()
final_lable_data = np.column_stack([y_true, y_pred, probs])
final_lable_data = pd.DataFrame(
    final_lable_data,
    columns = ["True_label", "Pre_label"] + [f"Class{i}_Prob" for i in range(25)],  # 列名
    index = MET500_samples  # 行名（样本名称）
)
final_lable_data.to_csv('/home/nanyuan/hdd/pathway_chat/method_compare/val_label/ourmethod/MET500_label.csv')

In [ ]:
#BCGSC
BCGSC_label_data = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/BCGSC/cancer_integrate1/label_data.csv')
BCGSC_exp_file  = '/home/nanyuan/hdd/pathway_chat/data/BCGSC/cancer_integrate1/exp_data_inte.csv'
BCGSC_exp_data = pd.read_csv(BCGSC_exp_file, index_col=0)
BCGSC_label_data['samples'] = BCGSC_label_data['samples'].astype(str)
BCGSC_exp_data.columns = BCGSC_exp_data.columns.str.replace('.', '-', regex=False)
BCGSC_label_dict  = dict(zip(BCGSC_label_data['samples'], BCGSC_label_data['label']))
BCGSC_dataset = ExpressionDataset(BCGSC_exp_data, BCGSC_label_dict)
BCGSC_dataloader = DataLoader(BCGSC_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_BCGSC = BCGSC_exp_data.shape[1] % args.batch_size
edge_index_b_BCGSC = []
for i in range(other_BCGSC): 
    offset = i * 258 
    edge_index_b_BCGSC.append(pathway_ind + offset) 
            
edge_index_b_BCGSC = torch.cat(edge_index_b_BCGSC, dim=1) 
edge_weight_b_BCGSC = pathway_weight.repeat(other_BCGSC)
edge_weight_b_BCGSC = edge_weight_b_BCGSC.float()

_,list_BCGSC_out,list_BCGSC_loss,list_BCGSC_true = test(
    model_max,BCGSC_exp_data, BCGSC_dataloader, edge_index_b,edge_index_b_BCGSC, edge_weight_b,edge_weight_b_BCGSC,args.batch_size,loss_fn)

logits_pred = np.vstack(list_BCGSC_out)
y_true = np.concatenate(list_BCGSC_true)
logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
y_true_onehot = np.zeros((len(y_true), 25))
for i, lab in enumerate(y_true):
    y_true_onehot[i, int(lab)] = 1
BCGSC_AUPRC = average_precision_score(y_true_onehot, probs, average="weighted") 
y_pred = np.argmax(probs, axis=1)
BCGSC_ACC = accuracy_score(y_true, y_pred)
BCGSC_MCC = matthews_corrcoef(y_true, y_pred)
BCGSC_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
BCGSC_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
BCGSC_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
BCGSC_loss = sum(list_BCGSC_loss)/len(list_BCGSC_loss)

print('BCGSC_AUPRC:',BCGSC_AUPRC)
print('BCGSC_ACC:',BCGSC_ACC)
print('BCGSC_Precision:',BCGSC_Precision)
print('BCGSC_Recall:',BCGSC_Recall)
print('BCGSC_F1:',BCGSC_F1)
print('BCGSC_loss:',BCGSC_loss)


In [ ]:

labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
df_cm = pd.DataFrame(cm, index=labels, columns=labels)
df_cm.to_csv('/home/nanyuan/hdd/pathway_chat/res/temp/BCGSC.csv')

In [ ]:
BCGSC_samples = BCGSC_exp_data.columns.to_numpy()
final_lable_data = np.column_stack([y_true, y_pred, probs])
final_lable_data = pd.DataFrame(
    final_lable_data,
    columns = ["True_label", "Pre_label"] + [f"Class{i}_Prob" for i in range(25)],  # 列名
    index = BCGSC_samples  # 行名（样本名称）
)
final_lable_data.to_csv('/home/nanyuan/hdd/pathway_chat/res/temp/BCGSC_label.csv')

In [ ]:
#GSE2109
GSE2109_label_data = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/GSE2109/label_data_inte.csv')
GSE2109_exp_file  = '/home/nanyuan/hdd/pathway_chat/data/GSE2109/exp_data.csv'
GSE2109_exp_data = pd.read_csv(GSE2109_exp_file, index_col=0)
GSE2109_label_dict  = dict(zip(GSE2109_label_data['samples'], GSE2109_label_data['label']))
GSE2109_dataset = ExpressionDataset(GSE2109_exp_data, GSE2109_label_dict)
GSE2109_dataloader = DataLoader(GSE2109_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_GSE2109 = GSE2109_exp_data.shape[1] % args.batch_size
edge_index_b_GSE2109 = []
for i in range(other_GSE2109): 
    offset = i * 258 
    edge_index_b_GSE2109.append(pathway_ind + offset) 
            
edge_index_b_GSE2109 = torch.cat(edge_index_b_GSE2109, dim=1) 
edge_weight_b_GSE2109 = pathway_weight.repeat(other_GSE2109)
edge_weight_b_GSE2109 = edge_weight_b_GSE2109.float()

In [ ]:
_,list_GSE2109_out,list_GSE2109_loss,list_GSE2109_true,final_embed = test(
    model_max,GSE2109_exp_data, GSE2109_dataloader, edge_index_b,edge_index_b_GSE2109, edge_weight_b,edge_weight_b_GSE2109,args.batch_size,loss_fn)

logits_pred = np.vstack(list_GSE2109_out)
y_true = np.concatenate(list_GSE2109_true)
logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
y_true_onehot = np.zeros((len(y_true), 25))
for i, lab in enumerate(y_true):
    y_true_onehot[i, int(lab)] = 1
GSE2109_AUPRC = average_precision_score(y_true_onehot, probs, average="weighted")   
y_pred = np.argmax(probs, axis=1)
GSE2109_ACC = accuracy_score(y_true, y_pred)
GSE2109_MCC = matthews_corrcoef(y_true, y_pred)
GSE2109_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
GSE2109_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
GSE2109_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
GSE2109_loss = sum(list_GSE2109_loss)/len(list_GSE2109_loss)

print('GSE2109_AUPR:',GSE2109_AUPRC)
print('GSE2109_ACC:',GSE2109_ACC)
print('GSE2109_Precision:',GSE2109_Precision)
print('GSE2109_Recall:',GSE2109_Recall)
print('GSE2109_F1:',GSE2109_F1)
print('GSE2109_loss:',GSE2109_loss)


In [ ]:
df = pd.DataFrame({
    "predict": y_pred,
    "true": y_true
})
df.to_csv('/home/nanyuan/hdd/pathway_chat/res/Met500_pred.csv')

In [ ]:
#IMvigor210
IMvig_label_data = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/IMvigor210/cancer_integrate1/label_data.csv')
IMvig_exp_file  = '/home/nanyuan/hdd/pathway_chat/data/IMvigor210/cancer_integrate1/exp_tpm.csv'
IMvig_exp_data = pd.read_csv(IMvig_exp_file, index_col=0)
IMvig_label_dict  = dict(zip(IMvig_label_data['samples'], IMvig_label_data['label']))
IMvig_dataset = ExpressionDataset(IMvig_exp_data, IMvig_label_dict)
IMvig_dataloader = DataLoader(IMvig_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_IMvig = IMvig_exp_data.shape[1] % args.batch_size
edge_index_b_IMvig = []
for i in range(other_IMvig): 
    offset = i * 258 
    edge_index_b_IMvig.append(pathway_ind + offset) 
            
edge_index_b_IMvig = torch.cat(edge_index_b_IMvig, dim=1) 
edge_weight_b_IMvig = pathway_weight.repeat(other_IMvig)
edge_weight_b_IMvig = edge_weight_b_IMvig.float()


_,list_IMvig_out,list_IMvig_loss,list_IMvig_true = test(
    model_max,IMvig_exp_data, IMvig_dataloader, edge_index_b,edge_index_b_IMvig, edge_weight_b,edge_weight_b_IMvig,args.batch_size,loss_fn)

logits_pred = np.vstack(list_IMvig_out)
y_true = np.concatenate(list_IMvig_true)
logits_t = torch.tensor(logits_pred, dtype=torch.float32)  
probs = F.softmax(logits_t, dim=1).detach().cpu().numpy()
y_true_onehot = np.zeros((len(y_true), 25))
for i, lab in enumerate(y_true):
    y_true_onehot[i, int(lab)] = 1
IMvig_AUPRC = average_precision_score(y_true_onehot, probs, average="weighted")
y_pred = np.argmax(probs, axis=1)
IMvig_ACC = accuracy_score(y_true, y_pred)
IMvig_MCC = matthews_corrcoef(y_true, y_pred)
IMvig_Precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
IMvig_Recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
IMvig_F1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
IMvig_loss = sum(list_IMvig_loss)/len(list_IMvig_loss)

print('IMvig_AUPRC:',IMvig_AUPRC)
print('IMvig_ACC:',IMvig_ACC)
print('IMvig_Precision:',IMvig_Precision)
print('IMvig_Recall:',IMvig_Recall)
print('IMvig_F1:',IMvig_F1)
print('IMvig_MCC:',IMvig_MCC)
print('IMvig_loss:',IMvig_loss)
